# Insights para um Serviço de Livros Digitais

Autor: Fabrício Camacho

Data: 18/04/2026

## 1. Introdução
A pandemia do coronavírus alterou drasticamente a rotina global, levando as pessoas a passarem mais tempo em casa e a dedicarem-se mais à leitura. Esse fenômeno impulsionou o crescimento de startups voltadas para o mercado de livros digitais e audiolivros.Neste projeto, analisaremos o banco de dados de um desses serviços competitivos para extrair informações valiosas sobre o acervo e o comportamento dos usuários. O banco de dados contém registros detalhados sobre:
- **Livros**: Títulos, datas de publicação e número de páginas;
- **Autores e Editoras**: Identificação dos criadores e das empresas publicadoras;
- **Interação dos Usuários**: Avaliações (notas) e resenhas textuais.

## 2. Objetivos do Estudo
O objetivo principal desta análise é explorar os dados disponíveis para gerar uma proposta de valor fundamentada para o lançamento de um novo produto no mercado de leitura

Para isso, realizaremos as seguintes tarefas técnicas utilizando consultas SQL:
- Quantificar o volume de livros publicados após o início do milênio ($01/01/2000$);
- Analisar a popularidade (número de resenhas) e a recepção (nota média) de cada obra;
- Identificar a editora líder em publicações de livros substanciais (mais de 50 páginas);
- Localizar o autor com a melhor reputação média entre os usuários, considerando uma base estatística relevante de avaliações;
- Avaliar o nível de engajamento dos usuários "super-leitores" através da média de resenhas escritas.

## 3. Conexão com o Banco de Dados
Para realizar nossas consultas, utilizaremos a biblioteca `sqlalchemy` para estabelecer a conexão com o banco de dados PostgreSQL na nuvem, e a biblioteca `pandas` para armazenar e exibir os resultados das nossas queries SQL em formato de DataFrame.

In [8]:
# Importar bibliotecas necessárias
import pandas as pd
from sqlalchemy import create_engine

# Dicionário com as credenciais e informações do banco de dados
db_config = {
    'user': 'practicum_student', # nome de usuário
    'pwd': 's65BlTKV3faNIGhmvJVzOqhs', # senha
    'host': 'rc1b-wcoijxj3yxfsf3fs.mdb.yandexcloud.net',
    'port': 6432, # porta de conexão
    'db': 'data-analyst-final-project-db' # nome do banco de dados
} 

# Criando a string de conexão formatada
connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(
    db_config['user'],
    db_config['pwd'],
    db_config['host'],
    db_config['port'],
    db_config['db']
)

# Estabelecendo a conexão (engine)
engine = create_engine(connection_string, connect_args={'sslmode':'require'})

print("Conexão com o banco de dados estabelecida com sucesso!")


Conexão com o banco de dados estabelecida com sucesso!


## 4. Exploração Inicial das Tabelas e Modelo de Dados

Antes de iniciarmos as análises profundas, é fundamental compreender a estrutura do nosso banco de dados e como as informações se conectam. Nosso banco de dados é composto por 5 tabelas principais, com a tabela `books` atuando como o núcleo central do nosso modelo:
- **`books`**: Contém as informações principais de cada livro (título, páginas, data de publicação). Ela se relaciona com as tabelas de dimensão de criação:
    - Relaciona-se com **`authors`** através da chave estrangeira `author_id`;
    - Relaciona-se com **`publishers`** através da chave estrangeira `publisher_id`;
- **`ratings`** e **`reviews`**: São tabelas que registram as interações dos usuários. Ambas se conectam à tabela central `books` através da chave estrangeira `book_id`. A tabela `ratings` armazena a nota quantitativa, enquanto a `reviews` armazena o texto da resenha.

Abaixo, executaremos consultas SQL simples para inspecionar as primeiras linhas de cada tabela e garantir que os dados estão formatados conforme o esperado.


In [9]:
# Lista com o nome de todas as tabelas do banco de dados
tabelas = ['books', 'authors', 'publishers', 'ratings', 'reviews']

# Loop para executar uma consulta SQL simples em cada tabela e exibir as 5 primeiras linhas
for tabela in tabelas:
    # A consulta em SQL puro sendo passada como string
    query_exploracao = f"SELECT * FROM {tabela} LIMIT 5;"
    
    # Executando a consulta e armazenando no Pandas apenas para visualização
    df_exploracao = pd.io.sql.read_sql(query_exploracao, con=engine)
    
    # Exibindo o resultado de forma organizada
    print(f"--- Primeiras 5 linhas da tabela: {tabela.upper()} ---")
    display(df_exploracao)
    print("\n" + "="*80 + "\n")
    

--- Primeiras 5 linhas da tabela: BOOKS ---


,book_id,author_id,title,num_pages,publication_date,publisher_id
0,1,546,'Salem's Lot,594,2005-11-01,93
1,2,465,1 000 Places to See Before You Die,992,2003-05-22,336
2,3,407,13 Little Blue Envelopes (Little Blue Envelope...,322,2010-12-21,135
3,4,82,1491: New Revelations of the Americas Before C...,541,2006-10-10,309
4,5,125,1776,386,2006-07-04,268




--- Primeiras 5 linhas da tabela: AUTHORS ---


,author_id,author
0,1,A.S. Byatt
1,2,Aesop/Laura Harris/Laura Gibbs
2,3,Agatha Christie
3,4,Alan Brennert
4,5,Alan Moore/David Lloyd




--- Primeiras 5 linhas da tabela: PUBLISHERS ---


,publisher_id,publisher
0,1,Ace
1,2,Ace Book
2,3,Ace Books
3,4,Ace Hardcover
4,5,Addison Wesley Publishing Company




--- Primeiras 5 linhas da tabela: RATINGS ---


,rating_id,book_id,username,rating
0,1,1,ryanfranco,4
1,2,1,grantpatricia,2
2,3,1,brandtandrea,5
3,4,2,lorichen,3
4,5,2,mariokeller,2




--- Primeiras 5 linhas da tabela: REVIEWS ---


,review_id,book_id,username,text
0,1,1,brandtandrea,Mention society tell send professor analysis. ...
1,2,1,ryanfranco,Foot glass pretty audience hit themselves. Amo...
2,3,2,lorichen,Listen treat keep worry. Miss husband tax but ...
3,4,3,johnsonamanda,Finally month interesting blue could nature cu...
4,5,3,scotttamara,Nation purpose heavy give wait song will. List...


## 5. Análise de Dados
### 5.1 - Livros publicados após 2000
**Objetivo:** Encontrar o número total de livros publicados após 1º de janeiro de 2000. Isso nos ajudará a entender se o catálogo foca em obras mais contemporâneas ou clássicas.

In [10]:
# Query para contar livros publicados após 01/01/2000
query_1 = '''
SELECT COUNT(book_id) AS total_livros_pos_2000
FROM books
WHERE publication_date > '2000-01-01';
'''

# Executando a query e armazenando no DataFrame
df_1 = pd.io.sql.read_sql(query_1, con=engine)

# Exibindo o resultado
display(df_1)


,total_livros_pos_2000
0,819


**Observação:** A consulta revelou que existem **$819$** livros publicados após o início do ano 2000 no banco de dados. Isso indica uma forte presença de literatura contemporânea no catálogo, o que pode ser um ponto atrativo para leitores que buscam obras e best-sellers mais recentes.

### 5.2 - Engajamento e Avaliação por Livro
**Objetivo:** Encontrar o número de resenhas de usuários e a nota média para cada livro. Isso nos permite identificar quais são os títulos mais populares (com mais resenhas) e os mais bem avaliados pela comunidade. Utilizaremos subconsultas para evitar a duplicação de dados (produto cartesiano) ao unir múltiplas tabelas de fatos.


In [11]:
# Query para buscar número de resenhas e nota média por livro
query_2 = '''
SELECT 
    b.title,
    COALESCE(rev.num_reviews, 0) AS num_reviews,
    ROUND(COALESCE(rat.avg_rating, 0), 2) AS avg_rating
FROM books b
LEFT JOIN (
    SELECT book_id, COUNT(review_id) AS num_reviews 
    FROM reviews 
    GROUP BY book_id
) rev ON b.book_id = rev.book_id
LEFT JOIN (
    SELECT book_id, AVG(rating) AS avg_rating 
    FROM ratings 
    GROUP BY book_id
) rat ON b.book_id = rat.book_id
ORDER BY num_reviews DESC
LIMIT 10;
'''

# Executando a query e armazenando no DataFrame
df_2 = pd.io.sql.read_sql(query_2, con=engine)

# Exibindo o resultado (limitado aos 10 com mais resenhas para não poluir a tela)
display(df_2)

,title,num_reviews,avg_rating
0,Twilight (Twilight #1),7,3.66
1,The Alchemist,6,3.79
2,The Da Vinci Code (Robert Langdon #2),6,3.83
3,The Glass Castle,6,4.21
4,The Hobbit or There and Back Again,6,4.13
5,The Road,6,3.77
6,Outlander (Outlander #1),6,4.13
7,The Curious Incident of the Dog in the Night-Time,6,4.08
8,Water for Elephants,6,3.98
9,Eat Pray Love,6,3.40


**Observações:** A análise de engajamento revela que grandes best-sellers (como *Twilight* e *The Alchemist*) lideram em volume de resenhas. No entanto, o livro mais comentado não é necessariamente o mais aclamado: *Twilight* possui média $3,66$, enquanto *The Glass Castle* atinge $4,21$ com quase o mesmo número de interações. Isso sugere que a base de usuários do aplicativo consome ativamente sucessos comerciais, mas mantém um critério de avaliação rigoroso.

### 5.3 - Volume de Publicações por Editora
**Objetivo:** Identificar a editora que publicou o maior número de livros com mais de $50$ páginas (excluindo folhetos). Esse insight nos ajudará a entender quais parceiros comerciais fornecem o maior volume de conteúdo substancial para a plataforma.


In [12]:
# Query para encontrar a editora com mais livros (> 50 páginas)
query_3 = '''
SELECT 
    p.publisher, 
    COUNT(b.book_id) AS total_books
FROM books b
JOIN publishers p ON b.publisher_id = p.publisher_id
WHERE b.num_pages > 50
GROUP BY p.publisher
ORDER BY total_books DESC
LIMIT 1;
'''

# Executando a query e armazenando no DataFrame
df_3 = pd.io.sql.read_sql(query_3, con=engine)

# Exibindo o resultado
display(df_3)


,publisher,total_books
0,Penguin Books,42


**Observações:** A análise revela que a **Penguin Books** é a principal fornecedora de conteúdo extenso para a plataforma, com $42$ títulos que ultrapassam $50$ páginas. Isso destaca a importância de manter boas parcerias estratégicas com grandes casas editoriais tradicionais para garantir um acervo robusto e de qualidade para os leitores mais dedicados.

### 5.4 - Autores Mais Aclamados
**Objetivo:** Identificar o autor com a nota média mais alta, considerando exclusivamente livros que possuem pelo menos $50$ avaliações. Esse filtro estatístico é crucial para evitarmos que um autor com apenas uma avaliação nota $5$ fique no topo do ranking, garantindo que o resultado reflita uma aprovação consistente da comunidade de leitores.


In [13]:
# Query para encontrar o autor com maior nota média (filtro: livros com >= 50 avaliações)
query_4 = '''
SELECT 
    a.author, 
    ROUND(AVG(r.rating), 2) AS avg_rating
FROM books b
JOIN authors a ON b.author_id = a.author_id
JOIN ratings r ON b.book_id = r.book_id
WHERE b.book_id IN (
    SELECT book_id
    FROM ratings
    GROUP BY book_id
    HAVING COUNT(rating_id) >= 50
)
GROUP BY a.author
ORDER BY avg_rating DESC
LIMIT 1;
'''

# Executando a query e armazenando no DataFrame
df_4 = pd.io.sql.read_sql(query_4, con=engine)

# Exibindo o resultado
display(df_4)


,author,avg_rating
0,J.K. Rowling/Mary GrandPré,4.29


**Observações:** A análise de satisfação consistente revelou que **J.K. Rowling/Mary GrandPré** é a autoria mais aclamada da plataforma, atingindo uma nota média de $4,29$ em obras com pelo menos $50$ avaliações. Esse dado comprova a força de franquias consolidadas (como Harry Potter com as capas contendo ilustrações de Mary GrandPré) não apenas em atrair leitores, mas em manter um altíssimo nível de satisfação, indicando que promover essas obras é uma aposta segura para engajar e reter usuários.

### 5.5 - Engajamento dos "Super-Leitores"
**Objetivo:** Encontrar o número médio de resenhas de texto escritas por usuários que avaliaram mais de $50$ livros. Essa métrica nos permite entender o comportamento dos nossos "super-leitores": será que a alta quantidade de avaliações quantitativas (notas) também se traduz em engajamento qualitativo (escrever resenhas)?


In [14]:
# Query para calcular a média de resenhas dos usuários mais ativos
query_5 = '''
SELECT 
    ROUND(AVG(review_count), 2) AS avg_reviews_by_top_users
FROM (
    SELECT 
        top_users.username, 
        COUNT(rev.review_id) AS review_count
    FROM (
        SELECT username
        FROM ratings
        GROUP BY username
        HAVING COUNT(rating_id) > 50
    ) top_users
    LEFT JOIN reviews rev ON top_users.username = rev.username
    GROUP BY top_users.username
) subquery;
'''

# Executando a query e armazenando no DataFrame
df_5 = pd.io.sql.read_sql(query_5, con=engine)

# Exibindo o resultado
display(df_5)


,avg_reviews_by_top_users
0,24.33


**Observações:** A análise demonstra que os usuários que avaliaram mais de $50$ livros escrevem, em média, **$24,33$ resenhas**. Isso indica que os "super-leitores" da plataforma não são apenas consumidores passivos que distribuem notas quantitativas, mas sim criadores de conteúdo ativos. Eles escrevem resenhas textuais para quase metade dos livros que avaliam, fomentando a comunidade e enriquecendo a plataforma com opiniões detalhadas.

## 6. Conclusões Finais e Proposta de Valor
Neste projeto, utilizamos consultas SQL para analisar o acervo literário e o comportamento de engajamento de uma base de usuários de um aplicativo de leitura. Esses dados serão utilizados para gerar uma proposta de valor para um novo produto. Com base nos resultados das nossas *queries*, podemos extrair os seguintes direcionamentos estratégicos:
- **Acervo Contemporâneo e Extenso:** A plataforma possui um catálogo atualizado ($819$ livros publicados após 2000) e conta com o suporte de parcerias gigantes, como a **Penguin Books**, que garante o fornecimento de leituras substanciais (mais de $50$ páginas);
- **O Equilíbrio entre o Popular e o Aclamado:** Embora os grandes best-sellers atraiam o maior volume de resenhas e atenção inicial, a lealdade e as maiores notas estão com obras e autores de qualidade inquestionável (como a série Harry Potter, de J.K. Rowling, com média $4,29$). O novo produto deve balancear a vitrine entre "os mais comentados" e "os mais bem avaliados";
- **Foco em Comunidade e Super-Leitores:** Identificamos que os usuários mais ativos geram um volume massivo de conteúdo orgânico (uma média de $24,33$ resenhas por super-usuário).

**Proposta de Valor para o Novo Produto:**
O novo aplicativo deve ser projetado não apenas como um leitor de livros, mas como uma **comunidade literária ativa**. Recomenda-se a criação de recursos de *gamificação* (como selos ou recompensas) para incentivar os leitores a escreverem mais resenhas, além de destacar selos de "Qualidade" para autores com nota média acima de 4.0, facilitando a curadoria para novos usuários. Manter contratos com editoras tradicionais como a Penguin será vital para sustentar os leitores mais vorazes.
